# TNBC Drug Response Predictor — 01 Data Audit (Colab-safe, Full Notebook)

**Notebook version: 2026-10-12h** — if your run does not print this version in §0, you are running a STALE copy. Re-upload from Drive.

**PRD:** v1.0 (2026-09-23) — GDSC ranking only / Laptop-Colab / <6 months  
**Goal of this notebook:** satisfy PRD DoD #1 → `intersection counts + MDA-MB-231/MCF7 present`, without OOM on the 938 MB expression matrix.

| File (local) | Size | Shape (approx) |
|---|---|---|
| `Model.csv` | 0.67 MB | 2154 rows × 49 cols |
| `OmicsExpressionTPMLogp1HumanAllGenes.csv` | **938.5 MB** (26Q3) | rows × [ProfileID, is_default_entry, ModelID + ~60k genes] (wide!) |
| `REPURPOSINGAUCMatrix.csv` | 10.8 MB | 727 × 1482 |
| `REPURPOSINGLog2IC50Matrix.csv` | 7.4 MB | 727 × 1482 |
| `REPURPOSINGLog2ViabilityCollapsedConditions.csv` | 0.7 MB | 11847 × 6 |
| `REPURPOSINGLog2ViabilityConditions.csv` | 2.2 MB | 33913 × 7 |

> **Why your original code fails:** `pd.read_csv(path, nrows=5)` still has to parse a header line with ~60k+ column names (~5–10 MB of header text) + build a 5×60k DataFrame in float64. On Colab free tier this hangs / OOMs / KeyboardInterrupt — exactly what you saw in `files.ipynb` with `chunksize=100`. Manual `files.upload()` also fails for ~1 GB files. This notebook downloads the big file via wget link (Section 2) and fixes both.

**How to use:**
1. `Runtime → Change runtime type → High-RAM` if available.
2. Run top-to-bottom. Small files → manual upload (Files panel). Large file (938 MB) → Section 2 wget link.
3. All heavy steps use `usecols` + `float32` + column-batching. Never load the full ~60k gene matrix at once.

In [ ]:
# ── 0. Environment + deps ──────────────────────────────────────────────
import sys, os, gc, csv, time, pathlib, shutil
print(sys.version)
!cat /proc/meminfo | head -3
!df -h /content | tail -1

# pyarrow + polars + duckdb make wide-CSV handling 5-10x faster / lighter
!pip -q install -U pyarrow polars duckdb scikit-learn 2>&1 | tail -2

import pandas as pd, numpy as np
print('pandas', pd.__version__, '| numpy', np.__version__)
NB_VERSION = '2026-10-12h (01-audit)'; print('notebook version:', NB_VERSION)

In [ ]:
# ── 1. Paths: /content first, Drive OPTIONAL ───────────────────────────
# YOUR SETUP: big expression file (938 MB) → Section 2 wget link → /content.
# Everything else (Model.csv, REPURPOSING*.csv, …) → Files panel upload (left sidebar).
# Drive is only a fallback for persistence across sessions.
# ── 1. Paths: Colab (/content) or LOCAL (./outputs_local) — auto-detected ──
# Colab setup: big file → Section 2 wget link → /content; small files → Files upload.
# Local setup: inputs in ./Data/ (flat), outputs go to ./outputs_local/.
import os
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
OUT = '/content' if ON_COLAB else os.path.abspath('outputs_local')
os.makedirs(OUT, exist_ok=True)
print('OUT =', OUT, '| ON_COLAB =', ON_COLAB)

if ON_COLAB:
    from google.colab import drive, files
    try:
        drive.mount(OUT + '/drive', force_remount=False)
        print('Drive connected (optional).')
    except Exception as e:
        print('Drive skipped (fine — using /content only):', str(e)[:120])
else:
    print('Local run: inputs expected in ./Data/ (Model.csv, REPURPOSING*.csv, expression CSV).')


CANDIDATE_DIRS = [
    OUT + '/',
    'Data',
    '.',
    '/content',
    OUT + '/drive/MyDrive/TNBC/data/raw',
    OUT + '/drive/MyDrive/TNBC',
    OUT + '/drive/MyDrive',
]

# Primary: 26Q3 AllGenes (938 MB; cols = ProfileID, is_default_entry, ModelID + genes).
# Older ProteinCoding file accepted as fallback (§8 auto-detects the layout).
LARGE_CANDIDATE_NAMES = [
    'OmicsExpressionTPMLogp1HumanAllGenes.csv',
    'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv',
]
SMALL_FILES = [
    'Model.csv',
    'REPURPOSINGAUCMatrix.csv',
    'REPURPOSINGLog2IC50Matrix.csv',
    'REPURPOSINGLog2ViabilityCollapsedConditions.csv',
    'REPURPOSINGLog2ViabilityConditions.csv',
]

def resolve(name_or_list):
    names = [name_or_list] if isinstance(name_or_list, str) else name_or_list
    for d in CANDIDATE_DIRS:
        for n in names:
            p = os.path.join(d, n)
            if os.path.exists(p):
                return p
    return None

print('Large expression file:', resolve(LARGE_CANDIDATE_NAMES))
for f in SMALL_FILES:
    print(f'{f:55s} ->', resolve(f))

# If small files missing, uncomment to upload manually (NOT for the 938 MB file — use the Section 2 wget link for it!):
# uploaded = files.upload()  # then !cp <name> /content/  (only small files)

## 2. Large-file download (fallback only — prefer Drive)

Your `!wget -O ... "https://storage.googleapis.com/..."` has 3 traps:
1. **Filename mismatch:** `-O OmicsExpressionTPMLogp1...` ≠ local `OmicsExpressionTRANSCRIPT_TPMLogp1...`. Code below accepts both.
2. **Double-encoded signature:** `%252F` in your URL is a double-encoded `/`. If you copied it from a browser, re-copy fresh from [depmap.org](https://depmap.org) → Downloads. Signatures **expire** (`Expires=1791893085`) → `403 Forbidden` means regenerate the link, don't retry.
3. **No resume / no check:** plain `wget` restarts a 938 MB download on disconnect. Always use `wget -c` (continue) + size check.

In [ ]:
# ── 2. Robust download (SKIP if file already resolved above) ────────────
import os, subprocess

EXPR_PATH = resolve(LARGE_CANDIDATE_NAMES)
if EXPR_PATH is None:
    EXPR_PATH = OUT + '/OmicsExpressionTPMLogp1HumanAllGenes.csv'
print('Target:', EXPR_PATH)

# ⟵ PASTE a FRESH URL here if needed (keep the single quotes!):
DEPMAP_URL = "PASTE_FRESH_URL_HERE"

if os.path.exists(EXPR_PATH) and os.path.getsize(EXPR_PATH) > 800_000_000:  # 26Q3 AllGenes ≈ 938 MB
    print(f'OK — already present ({os.path.getsize(EXPR_PATH)/1e9:.2f} GB), skipping download.')
elif DEPMAP_URL.startswith('http'):
    # -c = resume, --content-disposition keeps real name, --progress=bar:force shows % in Colab
    print('Downloading (resumable)... this takes ~5-15 min on Colab.')
    ret = subprocess.run(
        ['wget', '-c', '--content-disposition', f'--output-document={EXPR_PATH}',
         '--progress=bar:force:dot:giga', DEPMAP_URL],
    )
    print('wget exit code:', ret.returncode)
    if os.path.exists(EXPR_PATH):
        print(f'Size now: {os.path.getsize(EXPR_PATH)/1e9:.3f} GB')
        if ret.returncode != 0 or os.path.getsize(EXPR_PATH) < 800_000_000:
            print('⚠️ Incomplete or 403 — your signed URL probably expired. Regenerate it from depmap.org → Downloads.')
else:
    print('⚠️ Large file not found and no URL pasted.')
    print('   Paste a FRESH signed URL from depmap.org → Downloads → OmicsExpressionTPMLogp1HumanAllGenes.csv (26Q3, ~938 MB).')
    print('   Signed URLs expire — generate it right before downloading, then run this cell.')

In [ ]:
# ── 3. Inspect the GIANT file WITHOUT pandas (fast, <10 MB RAM) ─────────
# Never do pd.read_csv(big, nrows=5) directly — header alone is ~237k names!
import os
EXPR_PATH = resolve(LARGE_CANDIDATE_NAMES) or OUT + '/OmicsExpressionTPMLogp1HumanAllGenes.csv'
print('Using:', EXPR_PATH, f"({os.path.getsize(EXPR_PATH)/1e9:.2f} GB)" if os.path.exists(EXPR_PATH) else '(MISSING)')

!ls -lh "{EXPR_PATH}"
# Fast line count (rows+header) without loading:
!wc -l "{EXPR_PATH}" 2>&1 | head -1

# Read ONLY the header line with stdlib csv (no pandas overhead):
import csv
with open(EXPR_PATH, 'r', newline='', encoding='utf-8', errors='replace') as f:
    reader = csv.reader(f)
    header = next(reader)
    print(f'n_cols_total = {len(header):,}')
    print('first 8 col names:', header[:8])
    print('last 3 col names:', header[-3:])
    # peek 1 data row, first 8 values only
    row0 = next(reader)
    print('first data row, first 8 values:', row0[:8])
    print('header_line_chars ≈', sum(len(c) for c in header) + len(header), '(~MBs — this is why pandas hangs)')

# Identify metadata columns (no gene data loaded).
# 26Q3 AllGenes layout: ['ProfileID','is_default_entry','ModelID', <genes...>].
# §8 auto-detects positions from the header — nothing is hardcoded here.
meta_idx = {name: i for i, name in enumerate(header[:10])}
print(meta_idx)

In [ ]:
# ── 4. YOUR CODE — FIXED ───────────────────────────────────────────────
# ❌ Old (hangs/OOMs): pd.read_csv('/content/OmicsExpressionTPMLogp1HumanAllGenes.csv', nrows=5)
# ✅ Fixed: read only first ~12 columns for the preview. Full width is never previewed.
import pandas as pd

df_sample = pd.read_csv(EXPR_PATH, nrows=5, usecols=range(12))  # metadata + first genes only
print(f"الأبعاد الأولية للعينة: {df_sample.shape}")  # your original print, kept
display(df_sample.head())
print(f'\nFull width on disk: ~60k+ cols (AllGenes). Preview above shows first 12 cols only — this is intentional.')

# Even lighter alternative (zero pandas):
# import csv
# with open(EXPR_PATH) as f:
#     r = csv.reader(f); h = next(r); print(len(h)); print(h[:6]); print([next(r)[:6] for _ in range(2)])

In [ ]:
# ── 5. Load SMALL files normally (manual upload / Drive is fine) ────────
import pandas as pd, os

def must_resolve(name):
    p = resolve(name)
    assert p is not None, f'MISSING: {name} — upload it via Files panel or Drive, then re-run.'
    return p

model_path = must_resolve('Model.csv')
auc_path   = must_resolve('REPURPOSINGAUCMatrix.csv')
ic50_path  = must_resolve('REPURPOSINGLog2IC50Matrix.csv')
collap_path= must_resolve('REPURPOSINGLog2ViabilityCollapsedConditions.csv')
vcond_path = must_resolve('REPURPOSINGLog2ViabilityConditions.csv')

model_df = pd.read_csv(model_path)
print('Model.csv:', model_df.shape, '| cols:', list(model_df.columns[:6]), '...')

auc_df  = pd.read_csv(auc_path)
ic50_df = pd.read_csv(ic50_path)
print('AUC matrix:', auc_df.shape, '| example cols:', list(auc_df.columns[:3]))
print('IC50 matrix:', ic50_df.shape)

collap_df = pd.read_csv(collap_path)
vcond_df  = pd.read_csv(vcond_path)
print('CollapsedConditions:', collap_df.shape, '| ViabilityConditions:', vcond_df.shape)
display(collap_df.head(3))
display(vcond_df.head(3))

In [ ]:
# ── 6. TNBC annotation (PRD FR-03) — name is NOT enough ─────────────────
breast = model_df[model_df['OncotreeLineage'] == 'Breast'].copy()
print(f'Breast lines: {len(breast)} (expect ~96)')
display(breast[['ModelID', 'StrippedCellLineName', 'CCLEName', 'OncotreePrimaryDisease']].head(10))

# Mandatory controls
mask = model_df['StrippedCellLineName'].isin(['MDAMB231', 'MDAMB468', 'MCF7'])
display(model_df[mask][['ModelID', 'CellLineName', 'StrippedCellLineName', 'CCLEName', 'OncotreePrimaryDisease']])
assert 'ACH-000768' in set(model_df['ModelID']), 'MDA-MB-231 (ACH-000768) MISSING!'
assert 'ACH-000019' in set(model_df['ModelID']), 'MCF7 (ACH-000019) MISSING!'
print('✅ MDA-MB-231 + MCF7 present in Model.csv')

# Minimal TNBC vs Luminal map (extend with literature list for thesis; PRD FR-03)
# Known TNBC: MDAMB231, MDAMB468, BT549, HCC1937, HS578T ... | Luminal control: MCF7, T47D, ZR751 ...
TNBC_KNOWN = {'MDAMB231','MDAMB468','BT549','HCC1937','HS578T','BT20','MDAMB436','HCC38','HCC1143','HCC70'}
LUMINAL_KNOWN = {'MCF7','T47D','ZR751','MDAMB134VI','KPL1','CAMA1','EFM19'}
def coarse_subtype(s):
    s = str(s).upper()
    if s in TNBC_KNOWN: return 'TNBC'
    if s in LUMINAL_KNOWN: return 'Luminal'
    if s in {'SKBR3','AU565','BT474','HCC1954'}: return 'HER2+'
    return 'other'
breast['coarse_subtype'] = breast['StrippedCellLineName'].apply(coarse_subtype)
print(breast['coarse_subtype'].value_counts())
breast[['ModelID','StrippedCellLineName','coarse_subtype']].to_csv(OUT + '/breast_lines_annotated.csv', index=False)
print('saved /content/breast_lines_annotated.csv')

In [ ]:
# ── 7. Ferroptosis / candidate-drug panel (from your files.ipynb) ───────
drugs = ['sulfasalazine','simvastatin','lovastatin','erastin','RSL3',
         'ML162','ML210','ferrostatin','deferoxamine','acetylcysteine','Z-VAD','VAD-FMK']
out = []
for d in drugs:
    hit = collap_df[collap_df['CompoundName'].str.contains(d, case=False, na=False)][['CompoundName','CompoundID']].drop_duplicates()
    out.append(hit)
import pandas as pd
panel = pd.concat(out) if out else pd.DataFrame()
display(panel)
ids = panel['CompoundID'].unique().tolist() if len(panel) else []
print('Found IDs:', ids)
print('Present in AUC matrix:', [c for c in auc_df.columns if c in ids])
print('Present in IC50 matrix:', [c for c in ic50_df.columns if c in ids])

In [ ]:
# ── 8. Expression metadata ONLY (light) → intersection audit ────
# Auto-detect layout: 26Q3 AllGenes = [ProfileID, is_default_entry, ModelID, genes...].
# Column counts are NEVER hardcoded — the header tells us where genes start.
import csv
import pandas as pd

with open(EXPR_PATH, newline='', encoding='utf-8', errors='replace') as f:
    hdr = next(csv.reader(f))
low = [h.strip().lower() for h in hdr]

def _find(*keys):
    for i, h in enumerate(low):
        if h and any(k in h for k in keys):
            return i
    return None

id_pos = _find('modelid')
# Prefer the MODEL-level flag (IsDefaultEntryForModel) over the model-condition
# flag (IsDefaultEntryForMC) when both exist — we want one row per cell LINE:
def_pos = _find('isdefaultentryformodel')
if def_pos is None:
    def_pos = _find('default')
assert id_pos is not None, f'ModelID column not found. Header starts with: {hdr[:8]}'
id_col, flag_col = hdr[id_pos], (hdr[def_pos] if def_pos is not None else None)
GENE_START = max([0, id_pos] + ([def_pos] if def_pos is not None else [])) + 1
# Ground-truth probe: the first DATA row tells us which leading columns are
# really numeric. Any leading string column (missed meta col) pushes GENE_START right.
# This makes an off-by-one layout IMPOSSIBLE regardless of header naming.
with open(EXPR_PATH, newline='', encoding='utf-8', errors='replace') as _pf:
    _pr = csv.reader(_pf); _ph = next(_pr); _prow = next(_pr)
def _isnum(_v):
    _v = _v.strip()
    if _v == '' or _v.lower() in {'na', 'nan', 'none', 'null'}:
        return True
    try:
        float(_v); return True
    except ValueError:
        return False
_num_start = 0
for _i, _v in enumerate(_prow[:12]):
    if not _isnum(_v):
        _num_start = _i + 1
print('probe first-row values:', _prow[:8])
print(f'name-based start: {GENE_START} | value-based start: {_num_start}')
GENE_START = max(GENE_START, _num_start)
assert GENE_START <= 10, f'GENE_START={GENE_START} absurd — inspect header: {hdr[:10]}'
print(f'ID col: {id_col} (pos {id_pos}) | flag col: {flag_col} | GENE_START = {GENE_START}')
print(f'total cols = {len(hdr):,} → genes ≈ {len(hdr) - GENE_START:,}')

# Light 1-2 column read (seconds): every file row's ID + default flag
use = sorted({id_pos} | ({def_pos} if def_pos is not None else set()))
file_idx = pd.read_csv(EXPR_PATH, usecols=use, engine='c', low_memory=True)
file_idx.columns = [hdr[i] for i in use]
print('file rows (incl. non-default):', len(file_idx))

def _is_default(s):
    return s.astype(str).str.strip().str.lower().isin({'true', 'yes', '1', 't', 'y'})

is_default = _is_default(file_idx[flag_col]) if flag_col else pd.Series(True, index=file_idx.index)
print(f'default-entry rows: {int(is_default.sum())} / {len(file_idx)}')
expr_meta = file_idx[is_default].drop_duplicates(subset=[id_col], keep='first').reset_index(drop=True)
print('expr_meta (one row per ModelID):', expr_meta.shape)
display(expr_meta.head(3))

# Positional mask over FILE rows (drives the batched gene loader): wanted line AND default entry.
# (Section 8 runs after Section 6, so `breast` already exists ✅)
breast_ids = set(breast['ModelID'].astype(str))
row_mask_full = (file_idx[id_col].astype(str).isin(breast_ids | {'ACH-000768', 'ACH-000019'}) & is_default).to_numpy()
print(f'wanted file rows: {int(row_mask_full.sum())} / {len(file_idx)}')

expr_ids = set(expr_meta[id_col].astype(str))
model_ids = set(model_df['ModelID'].astype(str))
auc_ids = set(auc_df.iloc[:, 0].astype(str))  # first col is ACH- ID (Unnamed: 0)
print(f"Expression IDs: {len(expr_ids)} | Model.csv: {len(model_ids)} | AUC rows: {len(auc_ids)} | Breast: {len(breast_ids)}")
print(f"Expression ∩ Model: {len(expr_ids & model_ids)}")
print(f"Expression ∩ AUC: {len(expr_ids & auc_ids)}")
print(f"Breast ∩ Expression: {len(breast_ids & expr_ids)}")
print(f"Breast ∩ AUC: {len(breast_ids & auc_ids)}")
print('MDA-MB-231 in expression:', 'ACH-000768' in expr_ids)
print('MCF7 in expression:', 'ACH-000019' in expr_ids)
print('MDA-MB-231 in AUC:', 'ACH-000768' in auc_ids)
print('MCF7 in AUC:', 'ACH-000019' in auc_ids)

# Save the breast-only ID list — drives the batched gene loader below
wanted_ids = sorted((breast_ids & expr_ids) | {'ACH-000768', 'ACH-000019'})
pd.Series(wanted_ids).to_csv(OUT + '/wanted_breast_ids.csv', index=False)
print(f'Wanted breast+control lines with expression: {len(wanted_ids)} → {OUT}/wanted_breast_ids.csv')

# Layout file for §9a/§9b (gene offset + id column + row count)
import json as _json
_json.dump({'gene_start': GENE_START, 'id_col': id_col, 'n_file_rows': len(file_idx),
            'file': os.path.basename(EXPR_PATH)},
           open(OUT + '/expr_layout.json', 'w'))
print('saved /content/expr_layout.json', open(OUT + '/expr_layout.json').read())


## 9. Memory-safe gene loading → variance filter → PCA-128 (PRD §8)

`Expression (19k genes) → variance filter top 2000 → log2(TPM+1)+z-score → PCA 128`.  
The file actually holds **transcript-level** columns (`ENST...`), ~237k. We stream it in **column batches** (e.g. 5000 cols at a time), keep only wanted breast rows, accumulate per-gene variance with Welford running stats — RAM stays <2 GB.  
If you only need the audit (PRD W1-2), you can stop after the variance table. For the gold dataset (W3-5), run the PCA cell too.

In [ ]:
# ── 9a. Batched variance scan (column blocks; NON-HOLDOUT breast rows only) ──
# Vectorized per-batch nanvar (no Python per-value loop) — minutes, not hours.
# Holdout (MDA-MB-231 + MCF7 + 20% extra, seed 42) is EXCLUDED here and saved
# to /content/holdout_lines.npy so §9b/§11 and notebook 02 use the SAME split.
import csv, numpy as np, pandas as pd, gc, json as _json

layout = _json.load(open(OUT + '/expr_layout.json'))
GS, ID_COL = layout['gene_start'], layout['id_col']
print('layout:', layout)

BATCH = 5000          # gene-columns per batch (tune down to 2000 if RAM < 12 GB)
wanted_ids = pd.read_csv(OUT + '/wanted_breast_ids.csv', header=None)[0].astype(str).tolist()
wanted = set(wanted_ids)
print(f'Tracking {len(wanted)} cell lines')

# Shared holdout — identical logic everywhere (seed 42)
HOLDOUT_FIXED = {'ACH-000768', 'ACH-000019'}  # MDA-MB-231 + MCF7 always holdout
rng = np.random.RandomState(42)
pool = sorted(wanted - HOLDOUT_FIXED)
extra = set(rng.choice(pool, size=max(1, len(pool) // 5), replace=False))
HOLDOUT = HOLDOUT_FIXED | set(extra)
np.save(OUT + '/holdout_lines.npy', np.array(sorted(HOLDOUT)))
print(f'HOLDOUT lines ({len(HOLDOUT)}): {sorted(HOLDOUT)[:6]}... → {OUT}/holdout_lines.npy')

# File-row masks (positions match the expression file row order)
file_ids = pd.read_csv(EXPR_PATH, usecols=[ID_COL], engine='c', low_memory=True)[ID_COL].astype(str).to_numpy()
is_wanted = np.array([v in wanted for v in file_ids])
is_hold = np.array([v in HOLDOUT for v in file_ids])
mask_var = is_wanted & ~is_hold   # variance + PCA-fit rows: wanted, NON-holdout
mask_all = is_wanted               # dataset rows: all wanted (holdout included, split later)
print(f'variance/fit rows: {mask_var.sum()} | dataset rows: {mask_all.sum()} / {len(file_ids)}')
assert mask_var.sum() > 0 and mask_all.sum() > 0
np.save(OUT + '/mask_var.npy', mask_var)
np.save(OUT + '/mask_all.npy', mask_all)

with open(EXPR_PATH, newline='', encoding='utf-8', errors='replace') as f:
    hdr = next(csv.reader(f))
gene_names = hdr[GS:]
G = len(gene_names)
print(f'Total gene cols: {G:,} (offset {GS})')

var = np.zeros(G, dtype=np.float64)
import time
t0 = time.time()
for start in range(0, G, BATCH):
    stop = min(start + BATCH, G)
    use = list(range(GS + start, GS + stop))  # file coordinates
    chunk = pd.read_csv(EXPR_PATH, usecols=use, dtype=np.float32, engine='c', low_memory=True)
    assert chunk.shape == (len(file_ids), stop - start), chunk.shape
    _bad = [c for c, t in zip(chunk.columns, chunk.dtypes) if t != np.float32]
    assert not _bad, ('Non-numeric column(s) in gene batch: %s — layout is off! '
                      'Delete /content/expr_layout.json + masks, re-run §8 (probe prints the real start), then §9a.' % _bad[:3])
    sub = chunk.to_numpy(dtype=np.float64, copy=False)[mask_var]  # (n_fit × batch)
    del chunk; gc.collect()
    # values are already logp1 ("Logp1" in filename) — no extra log needed
    with np.errstate(invalid='ignore'):
        var[start:stop] = np.nanvar(sub, axis=0, ddof=1)
    if (start // BATCH) % 5 == 0:
        print(f'  batch {start:,}-{stop:,} / {G:,}  ({time.time()-t0:.0f}s)', flush=True)

var = np.where(np.isnan(var), 0, var)
order = np.argsort(var)[::-1]
TOP_N = 2000
top_idx = order[:TOP_N]
top_genes = [gene_names[i] for i in top_idx]
print(f'Top-2000 variance range: {var[top_idx[0]]:.3f} … {var[top_idx[-1]]:.3f}')
pd.DataFrame({'gene': top_genes, 'var': var[top_idx]}).to_csv(OUT + '/top2000_genes.csv', index=False)
np.save(OUT + '/top2000_idx.npy', top_idx)
print('saved /content/top2000_genes.csv + top2000_idx.npy')


In [ ]:
# ── 9b. Load Top-2000 × wanted-rows matrix → z-score → PCA-128 ──────────
# Leakage-proof: StandardScaler + PCA are FIT on non-holdout breast rows ONLY,
# then applied (transform) to all wanted rows. Split happens later — features for all.
import numpy as np, pandas as pd, gc, json as _json
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

layout = _json.load(open(OUT + '/expr_layout.json'))
GS = layout['gene_start']
mask_var = np.load(OUT + '/mask_var.npy')
mask_all = np.load(OUT + '/mask_all.npy')
file_ids = pd.read_csv(EXPR_PATH, usecols=[layout['id_col']], engine='c', low_memory=True)[layout['id_col']].astype(str).to_numpy()
assert len(file_ids) == len(mask_var) == len(mask_all), 'mask/file row mismatch — re-run §8→§9a in order!'
# One row per ModelID (first occurrence in file order): duplicate default rows
# overweight the PCA fit AND break the predictor's len==1 assert downstream.
_f = __import__('pandas').DataFrame({'id': file_ids, 'pos': __import__('numpy').arange(len(file_ids)), 'var': mask_var})
_f = _f[mask_all].drop_duplicates(subset=['id'], keep='first')
mask_all = __import__('numpy').zeros(len(file_ids), dtype=bool); mask_all[_f['pos'].to_numpy()] = True
mask_var = mask_var & mask_all
print(f'deduped dataset rows: {int(mask_all.sum())} (fit rows: {int(mask_var.sum())})') — re-run §8→§9a in order!'

top_idx = np.load(OUT + '/top2000_idx.npy')
file_cols = sorted((GS + int(i) for i in top_idx))  # file coordinates, sorted for C parser
pos_of = {fc: k for k, fc in enumerate(file_cols)}
order_back = [pos_of[GS + int(i)] for i in top_idx]  # reorder to variance-ranked order

mat = pd.read_csv(EXPR_PATH, usecols=file_cols, dtype=np.float32, engine='c').to_numpy(dtype=np.float32)
print('full-cohort top2000 matrix:', mat.shape)
X_fit = mat[mask_var][:, order_back]     # fit ONLY here (no holdout inside ✅)
X_all = mat[mask_all][:, order_back]     # transform everything (dataset rows)
ids_all = file_ids[mask_all]
del mat; gc.collect()
print('fit rows:', X_fit.shape, '| dataset rows:', X_all.shape)

# median-impute: median computed on FIT rows only (PRD FR-02)
miss = np.isnan(X_fit).mean(axis=0)
keep = miss <= 0.2
print(f'genes kept (≤20% missing): {keep.sum()}/{len(keep)}')
X_fit, X_all = X_fit[:, keep], X_all[:, keep]
med = np.nanmedian(X_fit, axis=0)
X_fit = np.where(np.isnan(X_fit), med, X_fit).astype(np.float32)
X_all = np.where(np.isnan(X_all), med, X_all).astype(np.float32)

N_COMP = int(min(128, X_fit.shape[0], X_fit.shape[1]))
print(f'PCA comps: {N_COMP} (min of 128, n_fit_samples, n_genes)')
scaler = StandardScaler().fit(X_fit)
pca = PCA(n_components=N_COMP, random_state=42)
pca.fit(scaler.transform(X_fit))
P = pca.transform(scaler.transform(X_all))
print(f'PCA-{N_COMP} explained variance: {pca.explained_variance_ratio_.sum():.3f}')

pca_df = pd.DataFrame(P, columns=[f'pca_{i}' for i in range(N_COMP)])
pca_df.insert(0, 'ModelID', ids_all)
pca_df.to_parquet(OUT + '/expression_pca128_breast.parquet', index=False)
print('saved /content/expression_pca128_breast.parquet', pca_df.shape)
display(pca_df.head(3))


In [ ]:
# ── 10. Build gold ML dataset: (pca_128 + drug_one_hot → auc, ln_ic50) ──
# PRD FR-04: one row per cell_line × drug pair.
import pandas as pd, numpy as np

pca_df = pd.read_parquet(OUT + '/expression_pca128_breast.parquet')
print('PCA:', pca_df.shape)

# Melt AUC wide → long (727 lines × 1482 drugs → ~1M pairs; breast slice ~140k)
auc_long = auc_df.melt(id_vars=[auc_df.columns[0]], var_name='drug_id', value_name='auc')
auc_long = auc_long.rename(columns={auc_df.columns[0]: 'ModelID'})
auc_long = auc_long.dropna(subset=['auc'])
auc_long = auc_long[(auc_long['auc'] >= 0) & (auc_long['auc'] <= 1)]  # Pandera-light (PRD §7)
print('AUC pairs (all):', len(auc_long))

# Optional: attach ln_ic50 as secondary target
ic50_long = ic50_df.melt(id_vars=[ic50_df.columns[0]], var_name='drug_id', value_name='ln_ic50')
ic50_long = ic50_long.rename(columns={ic50_df.columns[0]: 'ModelID'})
ml = auc_long.merge(ic50_long, on=['ModelID', 'drug_id'], how='left')
ml = ml.merge(pca_df, on='ModelID', how='inner')  # keep only breast+controls with expression
print('ML rows (breast ∩ AUC):', len(ml), '| cell lines:', ml['ModelID'].nunique(), '| drugs:', ml['drug_id'].nunique())

# Drug one-hot (GDSC/REPURPOSING ~hundreds of drugs; keep sparse-friendly uint8)
drug_dummies = pd.get_dummies(ml['drug_id'], prefix='drug', dtype=np.uint8)
ml_full = pd.concat([ml[['ModelID', 'drug_id', 'auc', 'ln_ic50'] + [c for c in pca_df.columns if c.startswith('pca_')]], drug_dummies], axis=1)
print('ML dataset shape:', ml_full.shape)

ml_full.to_parquet(OUT + '/tnbc_ml_dataset.parquet', index=False)
print('saved /content/tnbc_ml_dataset.parquet')
# Optional Drive backup (skipped if you clicked 'No thanks'):
!if [ -d "/content/drive/MyDrive" ]; then cp /content/tnbc_ml_dataset.parquet "/content/drive/MyDrive/TNBC_tnbc_ml_dataset.parquet" && echo drive-backup-ok; else echo no-drive-skip; fi
!if [ -d "/content/drive/MyDrive" ]; then cp /content/expression_pca128_breast.parquet "/content/drive/MyDrive/TNBC_expression_pca128_breast.parquet" && echo drive-backup-ok; else echo no-drive-skip; fi
display(ml_full.head(3))

In [ ]:
# ── 11. Baseline + GroupKFold sanity (PRD FR-05/06) ─────────────────────
# Must beat DummyMean. Holdout: MDA-MB-231 + MCF7 + 20% TNBC lines fully unseen.
import numpy as np, pandas as pd, os
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

ml_full = pd.read_parquet(OUT + '/tnbc_ml_dataset.parquet')
# NOTE: exclude the 'drug_id' STRING column ('drug_id'.startswith('drug_') is True!) — features are pca_* + drug_DPC-* dummies only
feat = [c for c in ml_full.columns if c.startswith('pca_') or (c.startswith('drug_') and c != 'drug_id')]
X, y, g = ml_full[feat].to_numpy(dtype=np.float32), ml_full['auc'].to_numpy(dtype=np.float32), ml_full['ModelID'].to_numpy()
print(f'X {X.shape} | y {y.shape} | groups {len(set(g))} cell lines')

HOLDOUT_FIXED = {'ACH-000768', 'ACH-000019'}  # MDA-MB-231 + MCF7 always holdout
_hp = OUT + '/holdout_lines.npy'
if os.path.exists(_hp):
    hold = set(np.load(_hp, allow_pickle=True).tolist()) | HOLDOUT_FIXED
    print('holdout loaded from §9a (SAME split as features ✅):', len(hold))
else:  # fallback: identical logic (seed 42) over ML lines
    rng = np.random.RandomState(42)
    rest = sorted(set(g) - HOLDOUT_FIXED)
    extra = set(rng.choice(rest, size=max(1, len(rest) // 5), replace=False))
    hold = HOLDOUT_FIXED | set(extra)
hold = {h for h in hold if h in set(g)}
te = np.array([x in hold for x in g]); tr = ~te
print(f'train {tr.sum():,} | holdout {te.sum():,} (lines: {len(hold)})')

dummy = DummyRegressor(strategy='mean').fit(X[tr], y[tr])
pred = dummy.predict(X[te])
print(f"DummyMean → MAE {mean_absolute_error(y[te], pred):.4f} | RMSE {mean_squared_error(y[te], pred)**0.5:.4f} | R² {r2_score(y[te], pred):.4f} | Spearman {spearmanr(y[te], pred).statistic:.4f}")

# Quick GroupKFold (2 splits, tiny forest — upgrade to XGBoost/CatBoost in 02_modeling notebook)
gkf = GroupKFold(n_splits=3)
for fold, (a, b) in enumerate(gkf.split(X[tr], y[tr], g[tr])):
    rf = RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=fold).fit(X[tr][a], y[tr][a])
    p = rf.predict(X[tr][b])
    print(f"fold{fold} RF50 → R² {r2_score(y[tr][b], p):.3f} | Spearman {spearmanr(y[tr][b], p).statistic:.3f}")
    break  # one fold for speed; full CV in modeling notebook
print('\nTarget for CatBoost (PRD): R²>0.25 and per-line Spearman>0.4 on breast slice.')

In [ ]:
# ── 12. PRD DoD checklist + exports ─────────────────────────────────────
import os
import pandas as pd  # needed for audit_summary.json block below
for p in [OUT + '/breast_lines_annotated.csv', OUT + '/wanted_breast_ids.csv',
          OUT + '/top2000_genes.csv', OUT + '/expression_pca128_breast.parquet',
          OUT + '/tnbc_ml_dataset.parquet']:
    print(('✅ ' if os.path.exists(p) else '❌ MISSING ') + p + (f" ({os.path.getsize(p)/1e6:.1f} MB)" if os.path.exists(p) else ''))

# PRD DoD #1 artifact: machine-readable audit summary (was print-only before)
import json as _json2
try:
    _ml = pd.read_parquet(OUT + '/tnbc_ml_dataset.parquet')
    _ann = pd.read_csv(OUT + '/breast_lines_annotated.csv')
    audit = {
        'expression_file': open(OUT + '/expr_layout.json').read(),
        'breast_lines_total': int(len(_ann)),
        'subtype_counts': {k: int(v) for k, v in _ann['coarse_subtype'].value_counts().items()},
        'wanted_with_expression': int(len(pd.read_csv(OUT + '/wanted_breast_ids.csv', header=None))),
        'ml_rows': int(len(_ml)),
        'ml_cell_lines': int(_ml['ModelID'].nunique()),
        'ml_drugs': int(_ml['drug_id'].nunique()),
        'mda_mb_231_present': bool('ACH-000768' in set(_ml['ModelID'])),
        'mcf7_present': bool('ACH-000019' in set(_ml['ModelID'])),
        'auc_min': float(_ml['auc'].min()), 'auc_max': float(_ml['auc'].max()),
    }
    _json2.dump(audit, open(OUT + '/audit_summary.json', 'w'), indent=1)
    print('saved /content/audit_summary.json')
    print(_json2.dumps(audit, indent=1))
except Exception as e:
    print('audit_summary skipped:', str(e)[:200])

print('\nPRD DoD #1: 01_data_audit shows intersection counts + MDA-MB-231/MCF7 present → see Sections 6 & 8 above ✅')
print('Next: 02_modeling.ipynb → XGBoost/CatBoost + NDCG@10/Recall@10 + MDA-MB-231 Top-20 + SHAP + FastAPI/Streamlit.')

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `wget` → `403 Forbidden` | Signed DepMap URL expired | Regenerate fresh link from depmap.org → Downloads; don't reuse old `Expires=` URLs |
| `wget` hangs at 0% | Double-encoded `%252F` or missing quotes | Paste URL inside `"..."` exactly as copied; use `wget -c` |
| `pd.read_csv(big, nrows=5)` hangs | ~60k-col header parse + wide frame | Use Section 4 (`usecols=range(12)`) — never full width |
| `files.upload()` fails for big file | Browser limit | Use the Section 2 wget link (signed URLs expire — generate fresh), or sync via Drive desktop |
| Colab RAM crash in §9a | `BATCH` too large | Lower `BATCH` 5000 → 2000, `Runtime → Restart`, re-run from §8 |
| `KeyboardInterrupt` in chunk loop (your `files.ipynb` §22) | Same wide-matrix issue on laptop | Run this notebook on Colab High-RAM; laptop can't hold 237k cols |
| Column name mismatch (`ModelID` vs `Unnamed: 0`) | AUC uses `Unnamed: 0` for ACH IDs | Code handles via `auc_df.columns[0]` — don't hardcode |

**Data versions to log in thesis (PRD FR-01):** DepMap `public-26q3-526b.11` (from your URL) + REPURPOSING release + this notebook's git hash.